In this file we clean drawings and descriptions. 

1. We exclude pilot/test participants that are included in the drawing and description datasets (BD001-BD024, TT000, etc.)
2. We add age column to drawing metadata file

In [23]:
import pandas as pd
import re
from pathlib import Path

In [24]:
# Load drawing, age, and transcript dfs
draw = pd.read_csv("../data/raw/mongo_output/highres/AllDescriptives_images_final_birch_run_v1.csv")
age = pd.read_csv("../data/clean/participant_age_table.csv")
transcript = pd.read_csv("../data/clean/half_and_full_utterance_features.csv")

In [25]:
# Define the set of participants to KEEP overall
# Children: BD025–BD124 (non-pilot participants only)
allowed_bd = {f"BD{num:03d}" for num in range(25, 125)}
# Adults: AD001–AD020
allowed_ad = {f"AD{num:03d}" for num in range(1, 21)}
allowed_ids = allowed_bd | allowed_ad

In [26]:
transcript = transcript.rename(columns={"SubID": "participant_id"})

In [27]:
# Drop everyone else from metadata and print which IDs were dropped 
draw_ids_all = set(draw['participant_id'].unique())
ids_to_drop_from_draw = sorted(draw_ids_all - allowed_ids)

#Sanity check: Which IDs were dropped?
print(f"IDs dropped from data: {ids_to_drop_from_draw}")
print(f"Total dropped from data: {len(ids_to_drop_from_draw)}")


#Now create the filtered drawing and description dataframes so we're not editing the orignal dataframes
draw_filtered = draw[draw['participant_id'].isin(allowed_ids)].copy()
desc_filtered = transcript[transcript['participant_id'].isin(allowed_ids)].copy()

# Sanity check: Which IDs were kept? (should be AD001–AD020 and BD025–BD124)
remaining_ids = sorted(set(draw_filtered['participant_id'].unique()))
print(f"IDs kept in data: {remaining_ids}")
print(f"Total kept in data: {len(remaining_ids)}")

IDs dropped from data: ['Aj001', 'BD004', 'BD005', 'BD008', 'BD010', 'BD011', 'BD012', 'BD013', 'BD014', 'BD015', 'BD016', 'BD017', 'BD018', 'BD019', 'BD020', 'BD021', 'BD022', 'BD023', 'BD024', 'Bd001', 'Bd002', 'Bd003', 'Bd006', 'Bd009', 'En001', 'En002', 'En003', 'TT000', 'TT001', 'bd007']
Total dropped from data: 30
IDs kept in data: ['AD001', 'AD002', 'AD003', 'AD004', 'AD005', 'AD006', 'AD007', 'AD008', 'AD009', 'AD010', 'AD011', 'AD012', 'AD013', 'AD014', 'AD015', 'AD016', 'AD017', 'AD018', 'AD019', 'AD020', 'BD025', 'BD026', 'BD027', 'BD028', 'BD029', 'BD030', 'BD031', 'BD032', 'BD033', 'BD034', 'BD035', 'BD036', 'BD037', 'BD038', 'BD039', 'BD040', 'BD041', 'BD042', 'BD043', 'BD044', 'BD045', 'BD046', 'BD047', 'BD048', 'BD049', 'BD051', 'BD052', 'BD053', 'BD054', 'BD056', 'BD057', 'BD058', 'BD059', 'BD060', 'BD061', 'BD062', 'BD063', 'BD065', 'BD066', 'BD067', 'BD068', 'BD069', 'BD070', 'BD071', 'BD072', 'BD073', 'BD074', 'BD075', 'BD076', 'BD077', 'BD078', 'BD079', 'BD080', 'B

In [28]:
draw_filtered.head()

,session_id,trial_num,category,participant_id,filename,submit_time,submit_date,submit_date_readable,start_time,trial_duration,num_strokes,draw_duration,mean_intensity,bounding_box,age
0,cdm_run_v81759172605291,6,an_octopus,BD074,/labs/vislearnlab/experiments/birch-draw/data/...,1.759173e+12,2025-09-29T19:04:40.738Z,2025-09-29 12:04:40.738000,1.759173e+12,34.975,4,7.257,0.024613,"(np.int64(1), np.int64(223), np.int64(0), np.i...",NaN
1,cdm_run_v81759172605291,7,a_shark,BD074,/labs/vislearnlab/experiments/birch-draw/data/...,1.759173e+12,2025-09-29T19:05:26.393Z,2025-09-29 12:05:26.393000,1.759173e+12,36.021,10,24.385,0.076929,"(np.int64(8), np.int64(203), np.int64(46), np....",NaN
2,cdm_run_v81759172605291,8,a_crab,BD074,/labs/vislearnlab/experiments/birch-draw/data/...,1.759173e+12,2025-09-29T19:05:57.754Z,2025-09-29 12:05:57.754000,1.759173e+12,23.680,6,20.661,0.054608,"(np.int64(0), np.int64(223), np.int64(0), np.i...",NaN
30,cdm_run_v81762975639154,2,square,BD120,/labs/vislearnlab/experiments/birch-draw/data/...,1.762976e+12,2025-11-12T19:28:22.420Z,2025-11-12 11:28:22.420000,1.762976e+12,23.426,10,18.034,0.013373,"(np.int64(55), np.int64(168), np.int64(55), np...",NaN
31,cdm_run_v81762975639154,3,shape,BD120,/labs/vislearnlab/experiments/birch-draw/data/...,1.762976e+12,2025-11-12T19:28:57.840Z,2025-11-12 11:28:57.840000,1.762976e+12,31.586,4,16.817,0.015545,"(np.int64(0), np.int64(221), np.int64(31), np....",NaN


In [29]:
desc_filtered.head()

,participant_id,target_category,draw_interference,utterance_interference,draw_exists,utterance_exists,version,utterance_ns,utterance_masked,draw_full_features,draw_half_features,utterance_full_visual_features,utterance_full_semantic_features,utterance_full_masked_filtered,utterance_half_masked_filtered,utterance_half_visual_features,utterance_half_semantic_features
0,AD001,crab,0,0,1,1,3.0,"It has two claws and two legs, and it is usual...","It has two claws and two legs, and it is usual...","antannae, pinchers, legs, shell","pinchers, legs, shell","pinchers, legs, color, eyes, small",crawl,"It has two claws and two legs, and it is usual...","It has two claws and two legs, and it is usual...","pinchers, legs, color, small",NaN
1,AD002,crab,0,0,1,1,3.0,"A crab also, a crab can be all sorts of colors...","Also, it can be all sorts of colors and those ...","shell, legs, pinchers, antannae, eyes","shell, legs","color, shell, no patterns, pinchers, legs, ant...","pinch, grab things, hard","Also, it can be all sorts of colors and those ...","Also, it can be all sorts of colors and those ...","color, shell, no patterns",NaN
2,AD003,crab,0,0,1,1,3.0,"Um a crab has like claws, and the like snap, t...","Um it has like claws, and they like snap, they...","shell, mouth, antannae, eyes, legs, pinchers","shell, antannae, eyes, pinchers","pinchers, shell","pinch, shellfish","it has like claws, and they like snap, they ma...","it has like claws, and they like snap, they ma...",pinchers,pinch
3,AD004,crab,0,0,1,1,3.0,"Okay, so a crab is an underwater sea creature ...","Okay, so it is an underwater sea creature and ...","shell, legs, mouth, pinchers, eyes, antannae","shell, legs","small, color, shell, legs, pinchers","live in shallow water, crawl, sea animal, hard","Okay, so it is an underwater sea creature and ...","Okay, so it is an underwater sea creature and ...","color, shell","live in shallow water, sea animal, hard"
4,AD005,crab,0,0,1,1,3.0,"It's a small creature and it has um like uh, I...","It's a small creature and it has um like uh, I...","shell, eyes, legs, pinchers","shell, legs","small, legs, pinchers",pinch,"It's a small creature and it has like , I thin...","It's a small creature and it has like , I thin...","small, legs",NaN


In [30]:

#Note: Rename columns in drawing df so they are in the format our embedding generator expects them to be in (image1 for image path, text1 for category)
draw_filtered = draw_filtered.rename(columns={"category": "text1", 
                                             "filename": "image1",}
                                           )
#If needded:Delete any unnecessary columns
#desc_filtered = desc_filtered.drop(columns=['version','Unnamed: 5', 'test_date',])




In [31]:
draw_filtered.head()

,session_id,trial_num,text1,participant_id,image1,submit_time,submit_date,submit_date_readable,start_time,trial_duration,num_strokes,draw_duration,mean_intensity,bounding_box,age
0,cdm_run_v81759172605291,6,an_octopus,BD074,/labs/vislearnlab/experiments/birch-draw/data/...,1.759173e+12,2025-09-29T19:04:40.738Z,2025-09-29 12:04:40.738000,1.759173e+12,34.975,4,7.257,0.024613,"(np.int64(1), np.int64(223), np.int64(0), np.i...",NaN
1,cdm_run_v81759172605291,7,a_shark,BD074,/labs/vislearnlab/experiments/birch-draw/data/...,1.759173e+12,2025-09-29T19:05:26.393Z,2025-09-29 12:05:26.393000,1.759173e+12,36.021,10,24.385,0.076929,"(np.int64(8), np.int64(203), np.int64(46), np....",NaN
2,cdm_run_v81759172605291,8,a_crab,BD074,/labs/vislearnlab/experiments/birch-draw/data/...,1.759173e+12,2025-09-29T19:05:57.754Z,2025-09-29 12:05:57.754000,1.759173e+12,23.680,6,20.661,0.054608,"(np.int64(0), np.int64(223), np.int64(0), np.i...",NaN
30,cdm_run_v81762975639154,2,square,BD120,/labs/vislearnlab/experiments/birch-draw/data/...,1.762976e+12,2025-11-12T19:28:22.420Z,2025-11-12 11:28:22.420000,1.762976e+12,23.426,10,18.034,0.013373,"(np.int64(55), np.int64(168), np.int64(55), np...",NaN
31,cdm_run_v81762975639154,3,shape,BD120,/labs/vislearnlab/experiments/birch-draw/data/...,1.762976e+12,2025-11-12T19:28:57.840Z,2025-11-12 11:28:57.840000,1.762976e+12,31.586,4,16.817,0.015545,"(np.int64(0), np.int64(221), np.int64(31), np....",NaN


Now we add age_months and age_yrs from available age df to drawing df

In [32]:
#Drop column called 'age' that was erroneously included in the raw data export
draw_filtered = draw_filtered.drop(columns=["age"])

In [33]:
#Merge age info into drawing dataframe
draw_filtered = draw_filtered.merge(
    age[["participant_id", "age_months", "age_yrs"]],
    on="participant_id",
    how="left"
)

In [34]:
draw_filtered.head()

,session_id,trial_num,text1,participant_id,image1,submit_time,submit_date,submit_date_readable,start_time,trial_duration,num_strokes,draw_duration,mean_intensity,bounding_box,age_months,age_yrs
0,cdm_run_v81759172605291,6,an_octopus,BD074,/labs/vislearnlab/experiments/birch-draw/data/...,1.759173e+12,2025-09-29T19:04:40.738Z,2025-09-29 12:04:40.738000,1.759173e+12,34.975,4,7.257,0.024613,"(np.int64(1), np.int64(223), np.int64(0), np.i...",31.0,2.59
1,cdm_run_v81759172605291,7,a_shark,BD074,/labs/vislearnlab/experiments/birch-draw/data/...,1.759173e+12,2025-09-29T19:05:26.393Z,2025-09-29 12:05:26.393000,1.759173e+12,36.021,10,24.385,0.076929,"(np.int64(8), np.int64(203), np.int64(46), np....",31.0,2.59
2,cdm_run_v81759172605291,8,a_crab,BD074,/labs/vislearnlab/experiments/birch-draw/data/...,1.759173e+12,2025-09-29T19:05:57.754Z,2025-09-29 12:05:57.754000,1.759173e+12,23.680,6,20.661,0.054608,"(np.int64(0), np.int64(223), np.int64(0), np.i...",31.0,2.59
3,cdm_run_v81762975639154,2,square,BD120,/labs/vislearnlab/experiments/birch-draw/data/...,1.762976e+12,2025-11-12T19:28:22.420Z,2025-11-12 11:28:22.420000,1.762976e+12,23.426,10,18.034,0.013373,"(np.int64(55), np.int64(168), np.int64(55), np...",93.0,7.77
4,cdm_run_v81762975639154,3,shape,BD120,/labs/vislearnlab/experiments/birch-draw/data/...,1.762976e+12,2025-11-12T19:28:57.840Z,2025-11-12 11:28:57.840000,1.762976e+12,31.586,4,16.817,0.015545,"(np.int64(0), np.int64(221), np.int64(31), np....",93.0,7.77


In [35]:
#Merge age info into transcript dataframe
desc_filtered = desc_filtered.merge(
    age[["participant_id", "age_months", "age_yrs"]],
    on="participant_id",
    how="left"
)

In [36]:
#Sanity check of drawing columns
draw_filtered.columns.unique()

Index(['session_id', 'trial_num', 'text1', 'participant_id', 'image1',
       'submit_time', 'submit_date', 'submit_date_readable', 'start_time',
       'trial_duration', 'num_strokes', 'draw_duration', 'mean_intensity',
       'bounding_box', 'age_months', 'age_yrs'],
      dtype='str')

In [37]:
#Sanity check of description columns
desc_filtered.columns.unique()

Index(['participant_id', 'target_category', 'draw_interference',
       'utterance_interference', 'draw_exists', 'utterance_exists', 'version',
       'utterance_ns', 'utterance_masked', 'draw_full_features',
       'draw_half_features', 'utterance_full_visual_features',
       'utterance_full_semantic_features', 'utterance_full_masked_filtered',
       'utterance_half_masked_filtered', 'utterance_half_visual_features',
       'utterance_half_semantic_features', 'age_months', 'age_yrs'],
      dtype='str')

Now let's save a csv of the metadata with ages columns

In [38]:
#Optional sort for readability
draw_filtered = draw_filtered.sort_values(by=["participant_id"])
desc_filtered = desc_filtered.sort_values(by=["participant_id"])

In [39]:
#Sanity check before saving

print(f"Rows: {len(draw_filtered)}")
print(f"Missing ages: {draw_filtered['age_months'].isna().sum()}")
print("Columns:", list(draw_filtered.columns))

display(draw_filtered.head())

# ---- Participants with missing ages ----
missing_age_df = draw_filtered[draw_filtered["age_months"].isna()]

if missing_age_df.empty:
    print("\nNo participants are missing age values.")
else:
    print("\nParticipants with missing age values:")

    print(
        missing_age_df[["participant_id"]]
        .drop_duplicates()
        .sort_values("participant_id")
    )

#As of now I see BD085 and BD097 have missing ages. They were lost in a lab error during data collection.
#We will need to exclude these participants from any age-based analyses.

Rows: 1096
Missing ages: 17
Columns: ['session_id', 'trial_num', 'text1', 'participant_id', 'image1', 'submit_time', 'submit_date', 'submit_date_readable', 'start_time', 'trial_duration', 'num_strokes', 'draw_duration', 'mean_intensity', 'bounding_box', 'age_months', 'age_yrs']


,session_id,trial_num,text1,participant_id,image1,submit_time,submit_date,submit_date_readable,start_time,trial_duration,num_strokes,draw_duration,mean_intensity,bounding_box,age_months,age_yrs
753,cdm_run_v81752869472107,6,a_seahorse,AD001,/labs/vislearnlab/experiments/birch-draw/data/...,1.752870e+12,2025-07-18T20:14:10.295Z,2025-07-18 13:14:10.295000,1.752870e+12,10.183,4,8.058,0.008371,"(np.int64(52), np.int64(142), np.int64(77), np...",229.0,19.13
756,cdm_run_v81752869472107,9,an_octopus,AD001,/labs/vislearnlab/experiments/birch-draw/data/...,1.752870e+12,2025-07-18T20:16:31.074Z,2025-07-18 13:16:31.074000,1.752870e+12,15.795,2,14.056,0.023079,"(np.int64(62), np.int64(198), np.int64(28), np...",229.0,19.13
752,cdm_run_v81752869472107,5,a_penguin,AD001,/labs/vislearnlab/experiments/birch-draw/data/...,1.752870e+12,2025-07-18T20:13:25.570Z,2025-07-18 13:13:25.570000,1.752870e+12,8.342,9,6.272,0.021345,"(np.int64(32), np.int64(210), np.int64(55), np...",229.0,19.13
751,cdm_run_v81752869472107,4,this_square,AD001,/labs/vislearnlab/experiments/birch-draw/data/...,1.752870e+12,2025-07-18T20:12:49.015Z,2025-07-18 13:12:49.015000,1.752870e+12,6.462,1,3.374,0.011858,"(np.int64(59), np.int64(164), np.int64(57), np...",229.0,19.13
750,cdm_run_v81752869472107,3,shape,AD001,/labs/vislearnlab/experiments/birch-draw/data/...,1.752870e+12,2025-07-18T20:12:39.029Z,2025-07-18 13:12:39.029000,1.752870e+12,12.635,1,10.317,0.014728,"(np.int64(1), np.int64(218), np.int64(25), np....",229.0,19.13



Participants with missing age values:
     participant_id
827           BD085
1071          BD097


In [40]:
#Export cleaned drawings as csv
output_path_draw = "../data/clean/drawing_highres_clean.csv"
draw_filtered.to_csv(output_path_draw, index=False)

print(f"Saved cleaned drawing data with ages to: {output_path_draw}")

Saved cleaned drawing data with ages to: ../data/clean/drawing_highres_clean.csv


In [41]:
# Export transcripts_raw_clean.csv — utterance_ns already present in half_and_full_utterance_features
desc_out = desc_filtered[["participant_id", "target_category", "utterance_interference", "utterance_exists", "utterance_ns", "utterance_masked", "utterance_half_masked_filtered"]].copy()
desc_out = desc_out.sort_values("participant_id").reset_index(drop=True)

output_path_desc = "../data/clean/transcripts_raw_clean.csv"
desc_out.to_csv(output_path_desc, index=False)
print(f"Saved {output_path_desc}: {desc_out.shape}")

Saved ../data/clean/transcripts_raw_clean.csv: (655, 7)
